# LangChain RAG with HANA Vector Store and Optional Chat History

This notebook demonstrates building a **Retrieval-Augmented Generation (RAG)** pipeline using:
- **LangChain LCEL** (LangChain Expression Language)
- **SAP HANA Cloud Vector Store** via the [`langchain-hana`](https://github.com/SAP/langchain-integration-for-sap-hana-cloud) package
- **GPT-4o** from SAP GenAI Hub
- **Optional chat history** for follow-up questions

## Prerequisites
- SAP HANA Cloud instance with vector engine enabled
- SAP AI Core with deployed embedding and chat models
- The `SAP_HELP_PUBLIC` table must be populated by running `vector-rag-embedding/python/LangChain_HANA_VectorStore_Embeddings.ipynb` first
- Environment variables configured in a `.env` file (see `.env-example`)

## Setup

### Dependencies
Ensure the libraries mentioned in the `requirements.txt` file are installed.

### Environment Variables
Copy `.env-example` to `.env` and fill in your SAP HANA Cloud and SAP AI Core credentials. See the embedding notebook's README for details on each variable.

## Initialization and Imports

We begin by importing necessary components and setting up the environment.


In [1]:
## Step 1: Setup and Imports
import os
from dotenv import load_dotenv, find_dotenv

from hdbcli import dbapi

from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_hana import HanaDB
from langchain_hana.utils import DistanceStrategy

from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from gen_ai_hub.proxy.langchain.openai import OpenAIEmbeddings

# Loads configuration from the nearest .env file (searches from CWD upward).
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path=dotenv_path, override=True)


True

## Set Up HANA Vector Store

In [2]:
## Step 2: Connect to HANA Vector Store
connection = dbapi.connect(
    address=os.environ.get("HANA_ADDRESS"),
    port=os.environ.get("HANA_PORT"),
    user=os.environ.get("HANA_USER"),
    password=os.environ.get("HANA_PASSWORD"),
    autocommit=True,
    encrypt=True,
    sslValidateCertificate=True,
)

# Establish connection to Generative AI Hub on AI Core for LLM access
proxy_client = get_proxy_client("gen-ai-hub")

# Initialize embeddings model (configurable via EMBEDDING_MODEL env var)
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
embedding_model = OpenAIEmbeddings(proxy_model_name=EMBEDDING_MODEL)

# Connect to HANA Vector Store
vdb = HanaDB(
    embedding=embedding_model,
    connection=connection,
    distance_strategy=DistanceStrategy.COSINE,
    table_name="SAP_HELP_PUBLIC",
)

# Set vector store as a retriever for retrieval related operations
retriever = vdb.as_retriever(search_kwargs={"k": 2})

## Prompt Template

In [3]:
# Define prompt
user_prompt = '''
Use the following context information to answer to user's query.
Here is some context: {context}

Based on the above context, answer the following query:
{question}

The answer tone has to be very professional in nature.

If you don't know the answer, politely say that you don't know.
'''

# Create langchain prompt object to fit into langchain pipeline
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("user", user_prompt)
])

## Chat History (Optional)

In-memory chat history stored per session. The history window is capped in the chain definition below to keep the last N messages.

In [4]:
store = {}

def get_by_session_id(session_id: str) -> InMemoryChatMessageHistory:
    """Retrieve or create chat history for a given session."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

## Define the LLM and RAG Chain

In [5]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

# Initialize LLM from Generative AI Hub
llm = ChatOpenAI(
    proxy_model_name='gpt-4o',
    proxy_client=proxy_client,
    max_tokens=2000,
    temperature=0.5
)

HISTORY_WINDOW = 4  # Number of recent messages to keep

def get_trimmed_history(x):
    """Return the last HISTORY_WINDOW messages from history."""
    history = x.get("history", [])
    return history[-HISTORY_WINDOW:]

# Define LangChain pipeline: retrieve context, inject history, prompt LLM, parse output
base_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": RunnablePassthrough(),
        "history": get_trimmed_history
    }
    | prompt
    | llm
    | StrOutputParser()
)

## Add Optional Message History

In [6]:
# Wrap the chain with history message processing runnable to add history messages to conversation if opted.
chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_by_session_id,
    input_messages_key="question",
    history_messages_key="history",
)

## Chat Function with History Toggle

In [7]:
def chat_with_rag(question: str, session_id: str, use_history: bool = True):
    """
    Function to decide if history messages to be processed based user's input
    """
    if use_history:
        return chain_with_history.invoke(
            {"question": question},
            config={"configurable": {"session_id": session_id}}
        )
    else:
        return base_chain.invoke({"question": question, "history": []})

## Run It!

In [8]:
if __name__ == "__main__":
    session_id = "user-123"
    print(chat_with_rag("What is HANA Cloud and can you summarize the context you received?", session_id, use_history=True))
    print()
    print("-+-" * 30)
    print()
    print(chat_with_rag("Can you repeat?", session_id, use_history=True))

    # Without history
    print()
    print("-+-" * 30)
    print()
    print(chat_with_rag("What was your previous answer?", session_id, use_history=False))

SAP HANA Cloud is a cloud-native platform designed to provide a centralized environment for accessing, storing, and processing enterprise data in real time. It combines the power and performance of SAP HANA with advanced capabilities for data storage, virtualization, and multi-model data processing. Additionally, it enables users to build applications and models using various languages, interfaces, and tools provided within the platform. SAP HANA Cloud supports efficient data management by allowing organizations to determine where and how data is stored and accessed, based on their performance requirements.

The context provided offers a high-level overview of the features and capabilities of SAP HANA Cloud, emphasizing its ability to streamline enterprise data operations and support robust application development. Should you require further details, I would be happy to assist.

-+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+-

SAP HANA Cloud i

## Summary

- This setup uses **SAP HANA Cloud Vector Store** for retrieval via the [`langchain-hana`](https://github.com/SAP/langchain-integration-for-sap-hana-cloud) package.
- Responses are generated using **GPT-4o** from OpenAI available on Generative AI Hub. Other models can be used as well as listed on [help.sap.com](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/supported-models).
- **Chat history** is optional and supports follow-up questions. Current implementation processes history messages in-memory. However, the same can be stored in files or database as well. Please refer to the [LangChain documentation](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html) for processing history messages.
- Implemented using **LangChain Expression Language (LCEL)** for composability.